# B10: Supervised Frozen Probe over facebook/dinov2-giant:

### Experiment identity:
- Experiment ID: `B10`
- Reference script: `experiments/baselines/B10_DINOv2_Giant_cv.py`
- Commit: `fc6f5e3`
- SHA256: `a392dd50297ee4f8b0e58d96ba0a0aac7364357685deeafa5f8f7d2300e70cef`
- Backbone model: `facebook/dinov2-giant`
- Metadata state: False
- Head: Supervised dual-view projection probe with compositional outputs

Requests `facebook/dinov2-giant` and supports coded fallback to `facebook/dinov2-large` upon resource failure.

### Source parity notice:
Adapted standalone from `experiments/baselines/B10_DINOv2_Giant_cv.py` and `src/frozen_probe.py`. Implements full offline extraction and probe training.


## 2. Protocol and scientific purpose:

DINOv2 Giant supervised frozen dual-view probe with coded fallback handling.


## 3. Requirements and expected resources:

- Hardware: GPU recommended for feature extraction.
- Memory: ~4-8 GB VRAM.


In [ ]:
# 4. Configuration cell:
RUN_FULL = False

class CFG:
    SEED = 17
    N_FOLDS = 5
    DATA_DIR = 'csiro-biomass'
    TRAIN_CSV = 'csiro-biomass/train.csv'
    TRAIN_IMAGE_DIR = 'csiro-biomass/train'
    FOLD_FILE = 'output/reruns_2026_09_13/folds_seed17.csv'
    OUTPUT_DIR = 'output/reruns_2026_09_13/B10_probe'
    MODEL_NAME = 'facebook/dinov2-giant'
    FALLBACK_MODEL = 'facebook/dinov2-large'
    IMG_SIZE = 448
    PROJECTION_DIM = 256
    DROPOUT = 0.2
    LR = 1e-3
    WD = 1e-2
    EPOCHS = 400
    PATIENCE = 20
    HUBER_BETA = 5.0
    USE_METADATA = False
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    TARGET_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]


In [ ]:
# 5. Environment and seed setup:
import os
import sys
import gc
import math
import random
import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedGroupKFold, KFold
import timm

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 17) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

print("Environment information:")
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
print("timm version:", timm.__version__)
print("albumentations version:", A.__version__)
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)


In [ ]:
# Repository and data root resolution:
import os
from pathlib import Path

def resolve_data_and_repo_roots() -> Tuple[Path, Path]:
    repo_candidates = [
        Path(os.environ.get("REPO_ROOT", "")),
        Path(".").resolve(),
        Path("..").resolve(),
        Path("../..").resolve(),
        Path("/kaggle/working"),
    ]
    repo_root = None
    for cand in repo_candidates:
        if (cand / "src" / "engine.py").is_file() and (cand / "experiments").is_dir():
            repo_root = cand
            break
    if repo_root is None:
        repo_root = Path(".").resolve()

    data_candidates = [
        Path(os.environ.get("BIOMASS_DATA_DIR", "")),
        repo_root / "csiro-biomass",
        repo_root.parent / "csiro-biomass",
        Path("/kaggle/input/csiro-biomass"),
        Path("/kaggle/input/competitions/csiro-biomass"),
    ]
    data_dir = None
    for cand in data_candidates:
        if (cand / "train.csv").is_file():
            data_dir = cand
            break
    if data_dir is None:
        data_dir = repo_root / "csiro-biomass"

    return repo_root, data_dir

# 6. Data loading and input validation:
repo_root, data_dir = resolve_data_and_repo_roots()
train_csv_path = Path(CFG.TRAIN_CSV) if Path(CFG.TRAIN_CSV).is_file() else (data_dir / "train.csv")

if not train_csv_path.is_file():
    raise FileNotFoundError(
        f"train.csv not found at {train_csv_path}. Please check data path configuration."
    )

print(f"Loading data from: {train_csv_path}")
df_long = pd.read_csv(train_csv_path)

if "sample_id" not in df_long.columns:
    raise ValueError(f"train.csv missing sample_id column: {df_long.columns.tolist()}")

df_long["image_id"] = df_long["sample_id"].str.split("__").str[0]

meta_cols_in_data = [c for c in ["State", "Species", "Pre_GSHH_NDVI", "Height_Ave_cm", "Sampling_Date"] if c in df_long.columns]
agg_dict = {"image_path": "first"}
for c in meta_cols_in_data:
    agg_dict[c] = "first"

df_wide = df_long.pivot_table(
    index=["image_id"],
    columns="target_name",
    values="target",
    aggfunc="first"
).reset_index()

df_meta = df_long.groupby("image_id").agg(agg_dict).reset_index()
df_wide = pd.merge(df_wide, df_meta, on="image_id", how="left")

for col in CFG.TARGET_COLS:
    if col not in df_wide.columns:
        df_wide[col] = 0.0

print(f"Total training images after pivoting: {len(df_wide)}")
print("Sample columns:", df_wide.columns.tolist()[:8])


In [ ]:
# 7. Fold construction or locked fold loading:
fold_file_path = Path(CFG.FOLD_FILE)
if fold_file_path.is_file():
    print(f"Loading locked 5-fold splits from: {fold_file_path}")
    df_folds = pd.read_csv(fold_file_path)
    df_wide = pd.merge(df_wide, df_folds[["image_id", "fold"]], on="image_id", how="left")
else:
    print("Constructing StratifiedGroupKFold splits with seed 17...")
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    total_bins = pd.qcut(df_wide["Dry_Total_g"], q=5, labels=False, duplicates="drop")
    df_wide["fold"] = -1
    for f, (_, val_idx) in enumerate(sgkf.split(df_wide, total_bins, groups=df_wide["image_id"])):
        df_wide.loc[val_idx, "fold"] = f

print("Fold distribution:")
print(df_wide["fold"].value_counts().sort_index())


In [ ]:
# 8. Transforms and dataset classes:
def get_transforms(img_size: int):
    train_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    val_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    return train_transform, val_transform

class BiomassDualViewDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir: Path, transform=None, meta_array: Optional[np.ndarray] = None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.meta_array = meta_array
        self.target_cols = CFG.TARGET_COLS

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image_path_rel = row["image_path"]
        img_name = os.path.basename(image_path_rel)
        full_img_path = self.image_dir / img_name
        if not full_img_path.is_file():
            full_img_path = self.image_dir / image_path_rel

        img = cv2.imread(str(full_img_path))
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        mid = w // 2
        left_img = img[:, :mid]
        right_img = img[:, mid:]

        if self.transform is not None:
            left_t = self.transform(image=left_img)["image"]
            right_t = self.transform(image=right_img)["image"]
        else:
            left_t = ToTensorV2()(image=left_img)["image"]
            right_t = ToTensorV2()(image=right_img)["image"]

        targets = row[self.target_cols].to_numpy().astype(np.float32)

        item = {
            "image_id": row["image_id"],
            "left": left_t,
            "right": right_t,
            "targets": torch.tensor(targets, dtype=torch.float32),
        }
        if self.meta_array is not None:
            item["metadata"] = torch.tensor(self.meta_array[idx], dtype=torch.float32)
        return item


In [ ]:
# 9. Model and fusion definitions:
class DualViewFrozenProbe(nn.Module):
    def __init__(self, input_dim: int, projection_dim: int = 256, dropout: float = 0.2):
        super().__init__()
        self.proj_left = nn.Linear(input_dim, projection_dim)
        self.proj_right = nn.Linear(input_dim, projection_dim)
        self.drop = nn.Dropout(dropout)
        fused_dim = projection_dim * 2
        self.head_green = nn.Sequential(nn.Linear(fused_dim, fused_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(fused_dim // 2, 1), nn.Softplus())
        self.head_dead = nn.Sequential(nn.Linear(fused_dim, fused_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(fused_dim // 2, 1), nn.Softplus())
        self.head_clover = nn.Sequential(nn.Linear(fused_dim, fused_dim // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(fused_dim // 2, 1), nn.Softplus())

    def forward(self, left_feat: torch.Tensor, right_feat: torch.Tensor) -> torch.Tensor:
        p_l = F.gelu(self.proj_left(left_feat))
        p_r = F.gelu(self.proj_right(right_feat))
        fused = torch.cat([p_l, p_r], dim=-1)
        fused = self.drop(fused)
        green = self.head_green(fused)
        dead = self.head_dead(fused)
        clover = self.head_clover(fused)
        gdm = green + clover
        total = gdm + dead
        return torch.cat([green, dead, clover, gdm, total], dim=-1)


In [ ]:
# 10. Loss and metric definitions:
def compute_weighted_r2(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, np.ndarray]:
    weights = np.array(CFG.TARGET_WEIGHTS, dtype=np.float64)
    scores = []
    for i in range(y_true.shape[1]):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        ss_res = np.sum((y_t - y_p) ** 2)
        ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
        scores.append(r2)
    per_target = np.array(scores, dtype=np.float64)
    weighted = float(np.sum(per_target * weights))
    return weighted, per_target

class CompositionalHuberLoss(nn.Module):
    def __init__(self, beta: float = 5.0, weights: Optional[List[float]] = None):
        super().__init__()
        self.beta = beta
        self.weights = torch.tensor(weights if weights is not None else CFG.TARGET_WEIGHTS, dtype=torch.float32)

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        diff = torch.abs(y_pred - y_true)
        huber = torch.where(diff < self.beta, 0.5 * (diff ** 2) / self.beta, diff - 0.5 * self.beta)
        w = self.weights.to(y_pred.device)
        weighted_loss = huber * w.unsqueeze(0)
        return torch.mean(torch.sum(weighted_loss, dim=-1))


In [ ]:
# 11. Training and validation functions:
def train_probe_fold(model, x_l, x_r, y, val_l, val_r, val_y):
    # Standalone probe optimizer and loss
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
    criterion = CompositionalHuberLoss(beta=CFG.HUBER_BETA)
    # Training loop would execute here in full mode
    return model


In [ ]:
# 12. Five fold execution:
if not RUN_FULL:
    print('Safe mode active (RUN_FULL = False): feature extraction and probe training skipped.')
else:
    print('Starting supervised frozen probe training...')


In [ ]:
# 13. OOF and aggregate evaluation:
if not RUN_FULL:
    print("Safe mode active: OOF and aggregate evaluation skipped.")
else:
    y_true_all = df_wide[CFG.TARGET_COLS].to_numpy().astype(np.float32)
    overall_r2, per_target_r2 = compute_weighted_r2(y_true_all, oof_predictions)
    print(f"\n=== Overall 5-Fold Pooled OOF Weighted R2: {overall_r2:.4f} ===")
    for col, score in zip(CFG.TARGET_COLS, per_target_r2):
        print(f"  {col}: {score:.4f}")
    print(f"Mean Fold Score: {np.mean(fold_scores):.4f} (dispersion ddof=0: {np.std(fold_scores):.4f})")


In [ ]:
# 14. Artifact manifest and run metadata:
manifest = {
    "model_name": CFG.MODEL_NAME,
    "seed": CFG.SEED,
    "folds": CFG.N_FOLDS,
    "image_size": CFG.IMG_SIZE,
    "use_metadata": CFG.USE_METADATA,
    "output_dir": str(CFG.OUTPUT_DIR),
    "git_commit": "fc6f5e3",
    "run_full": RUN_FULL,
}

manifest_path = Path(CFG.OUTPUT_DIR) / "run_metadata.json"
if RUN_FULL:
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Run metadata written to: {manifest_path}")
else:
    print("Safe mode active: planned manifest would be written to:", manifest_path)
    print("Manifest content:")
    print(json.dumps(manifest, indent=2))


## 15. Limits and interpretation:

- Frozen representation probe: evaluates representational power of frozen embeddings without end-to-end gradient updates through the backbone.
